In [ ]:
import os
import numpy as np
import pandas as pd
import zipfile
from matplotlib import pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
from google.colab import drive
import pickle

In [ ]:
drive.mount('/content/drive')

In [ ]:
def collect_results(task, dates=["2025-02-15"]):

    num_epochs = 10

    results = {}
    for date in dates:
        param_set = os.listdir(f'/content/drive/MyDrive/master_data/{date}/{task}')
        for param in param_set:
            if os.path.isdir(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}'):
                results[param] = {}
                try:
                    results[param]["intersections"] = np.load(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}/metrics/intersections.npy', allow_pickle=True).flatten()
                except:
                    print(f'Error loading intersections for {param} and task {task}')
                try:
                    results[param]["reg_ratio"] = np.load(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}/metrics/reg_ratio.npy', allow_pickle=True)
                except:
                    print(f'Error loading reg_ratio for {param} and task {task}')
                try:
                    results[param]['avg_metric'] = np.load(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}/metrics/avg_metric.npy', allow_pickle=True),
                    results[param]['avg_metric'] = results[param]['avg_metric'][0]
                except:
                    print(f'Error loading avg_metric for {param} for task {task}')
    return results

In [ ]:
res_mis = collect_results('mis', dates=['2025-03-30'])
res_mds = collect_results('mds')
res_mc = collect_results('maxcut')

In [ ]:
def compute_metric_summary(metric_name, input_dict):
    results_summary = []
    for key, data in input_dict.items():
        metric_data = data[metric_name]
        summary = {
            'parameters': key,
            'mean': metric_data.mean(axis=0),
            'median': np.median(metric_data, axis=0),
            'std': metric_data.std(axis=0),
            'min': metric_data.min(axis=0),
            'max': metric_data.max(axis=0),
        }
        results_summary.append(summary)

    return pd.DataFrame(results_summary)

In [ ]:
# import shutil
# shutil.rmtree('/content/common')

In [ ]:
%mkdir /content/common/

Training process overview

In [ ]:
selected_params = [
    "frac_0_05_boost_50",
    "frac_0_10_boost_50",
    "frac_0_25_boost_50",
    "frac_0_50_boost_50",
    "frac_0_00_boost_1",
]

results_list = [res_mis, res_mds, res_mc]
tasks = ["MIS", "MDS", "MC"]

num_ticks = 11  # Number of ticks (11 for 0 to 10 inclusive)

for i, results in enumerate(results_list):
    plt.figure(figsize=(7, 5))  # new figure for each plot

    for param in selected_params:
        y = results[param].get('avg_metric', [0])
        param_label = f"f=0.{param.split('_')[2]} b={param.split('_')[-1]}"
        plt.plot(y, label=param_label)

    plt.title(f"{tasks[i]}: average reward over training process")
    plt.ylabel('Value')
    plt.xlabel('Epoch')

    y_len = len(y)
    x_ticks = np.linspace(0, y_len - 1, num_ticks)
    plt.xticks(x_ticks, np.linspace(0, 10, num_ticks).astype(int))

    plt.legend()
    plt.tight_layout()
    plt.savefig(f'/content/common/{tasks[i]}_training_process.png', dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
for task, results in zip(tasks, results_list):
    r = compute_metric_summary('intersections', results)
    if task != 'MIS':
        r = r[r["parameters"].isin(selected_params)]
    r["parameters"] = r["parameters"].apply(lambda x: f"f=0.{x.split('_')[2]} b={x.split('_')[-1]}")
    r.sort_values(by='mean', ascending=False).reset_index(drop=True).to_latex(
        f'/content/common/{task}_params_intersections.tex',
        float_format="%.2f",
        caption=f"Intersection metric on {task} task.",
        label=f"tab:params_intersection_{task}",
        column_format="l|ccccc",
        bold_rows=True,
        index=False,
    )


In [ ]:
for task, results in zip(tasks, results_list):
    r = compute_metric_summary('reg_ratio', results)
    if task != 'MIS':
        r = r[r["parameters"].isin(selected_params)]
    r["parameters"] = r["parameters"].apply(lambda x: f"f=0.{x.split('_')[2]} b={x.split('_')[-1]}")
    r.sort_values(by='mean', ascending=False).reset_index(drop=True).to_latex(
        f'/content/common/{task}_params_rr.tex',
        float_format="%.2f",
        caption=f"Regularization ratio metric on {task} task.",
        label=f"tab:params_rr_{task}",
        column_format="l|ccccc",
        bold_rows=True,
        index=False,
    )

In [ ]:
for task, results in zip(tasks, results_list):
    try:
        r = compute_metric_summary('avg_metric', results)
    except:
        print(f'Error computing avg_metric for {task}')
        continue
    if task != 'MIS':
        r = r[r["parameters"].isin(selected_params)]
    r["parameters"] = r["parameters"].apply(lambda x: f"f=0.{x.split('_')[2]} b={x.split('_')[-1]}")
    r.sort_values(by='mean', ascending=False).reset_index(drop=True).to_latex(
        f'/content/common/{task}_params_metric.tex',
        float_format="%.2f",
        caption=f"Scores on {task} task.",
        label=f"tab:params_scores_{task}",
        column_format="l|ccccc",
        bold_rows=True,
        index=False,
    )

In [ ]:
df_diff_scores = pd.DataFrame()

for task in ['MIS', 'MDS', 'MaxCut']:
    try:
        df_in = pd.read_csv(f"/content/{task.lower()}_dropdowns.csv")
        df_in["Task"] = task
        df_diff_scores = pd.concat([df_diff_scores, df_in])
    except:
        print(f"Error loading {task} scores diff")
df_diff_scores.set_index(["Metric", "Task"]).sort_index().to_latex(
        f'/content/common/lhs_score_dropdowns.tex',
        float_format="%.2f",
        caption=f"Relative difference of Likelihood and Scores comparing to canonical GFN.",
        label=f"tab:lhs_scores_dropdowns",
        column_format="lcc|ccccc",
        bold_rows=True,
    )

In [ ]:
df_diff_scores = pd.DataFrame()

for task in ['MIS', 'MDS', 'MaxCut']:
    try:
        df_in = pd.read_csv(f"/content/{task.lower()}_dropdowns_intersection.csv")
        df_in["Task"] = task
        df_diff_scores = pd.concat([df_diff_scores, df_in])
    except:
        print(f"Error loading {task} scores diff")
df_diff_scores.set_index(["Metric", "Task"]).sort_index().to_latex(
        f'/content/common/intersection_dropdowns.tex',
        float_format="%.2f",
        caption=f"Relative difference of Intersections comparing to greedy heuristic.",
        label=f"tab:intersections_dropdowns",
        column_format="lcc|ccccc",
        bold_rows=True,
    )